# 05 多通道卷积计算

前面已经学习了卷积层的基本运算规则：卷积核在图片上滑动，对应位置相乘再求和。

但是前面的例子大多默认输入是灰度图，也就是只有 1 个通道。

这一节继续往前走一步：如果输入不是 1 个通道，而是多个通道，卷积应该怎么算？

这一节仍然不写代码，只把计算规则讲清楚。

## 1. 为什么会有多通道

上一节我们已经知道，图像可以有通道。

灰度图只有 1 个通道，因为每个像素只需要一个数字表示明暗。

RGB 彩色图有 3 个通道，因为每个像素需要红、绿、蓝三个颜色分量。

可以这样记：

```text
灰度图：1 x H x W
RGB 图：3 x H x W
```

这里的 1 和 3，就是输入通道数。

## 2. 单通道卷积先回顾一下

如果输入是灰度图，只有 1 个通道。

假设卷积核大小是 3 x 3，那么卷积核也是一个 3 x 3 的小矩阵。

它每次盖住图片中的一个 3 x 3 局部区域，然后做：

```text
对应位置相乘，再全部加起来
```

这会得到一个输出数字。

卷积核滑完整张图之后，就得到一张特征图。

所以单通道卷积可以先理解成：

```text
一层输入区域 x 一层卷积核 -> 一个输出值
```

## 3. 多通道卷积的核心规则

多通道卷积最重要的一句话是：

```text
输入有几个通道，一个卷积核就要有几层。
```

比如输入是 RGB 图，有 3 个通道。

如果卷积核空间大小是 3 x 3，那么这个卷积核不是简单的一张 3 x 3 小表，而是 3 张 3 x 3 小表叠起来。

可以理解成：

```text
输入：R 通道 + G 通道 + B 通道

卷积核：R 对应一层权重
        G 对应一层权重
        B 对应一层权重
```

所以一个 RGB 输入上的 3 x 3 卷积核，实际形状可以理解成：

$$
3\times3\times3
$$

第一个 3 表示输入通道数，后面的 3 x 3 表示卷积核在高和宽方向的大小。

## 4. 多通道的一次卷积怎么算

假设输入是 RGB 图。

卷积核在某个位置盖住了三个局部区域：

```text
R 通道上的 3 x 3 区域
G 通道上的 3 x 3 区域
B 通道上的 3 x 3 区域
```

卷积核自己也有三层权重：

```text
R 通道对应的 3 x 3 权重
G 通道对应的 3 x 3 权重
B 通道对应的 3 x 3 权重
```

计算时分三步：

1. R 通道局部区域和 R 层卷积核对应位置相乘再求和。
2. G 通道局部区域和 G 层卷积核对应位置相乘再求和。
3. B 通道局部区域和 B 层卷积核对应位置相乘再求和。

最后，把三个通道算出来的结果再加起来。

可以写成：

$$
输出值 = R通道结果 + G通道结果 + B通道结果
$$

如果有偏置，再加上偏置：

$$
输出值 = R通道结果 + G通道结果 + B通道结果 + b
$$

## 5. 一个卷积核最后输出几张特征图

这里很容易误会。

输入是 RGB，有 3 个通道。

一个卷积核也有 3 层。

但这并不表示一个卷积核会输出 3 张特征图。

一个卷积核最终只输出 1 张特征图。

因为它会把所有输入通道上的计算结果加在一起，合成一个输出值。

所以要记住：

```text
一个卷积核 -> 一张输出特征图 -> 一个输出通道
```

输入通道数决定卷积核有多深。

卷积核个数决定输出通道数有多少。

## 6. 多个卷积核会怎样

如果只有 1 个卷积核，就只能得到 1 张特征图。

如果有 16 个卷积核，就会得到 16 张特征图。

也就是说：

```text
16 个卷积核 -> 16 张特征图 -> 16 个输出通道
```

这也是 CNN 里通道数会变化的原因。

比如输入是一张 RGB 图片：

$$
3\times32\times32
$$

如果使用 16 个卷积核，padding 和 stride 让高宽保持不变，那么输出就可以是：

$$
16\times32\times32
$$

注意：这里的 16 不是原图的颜色通道，而是模型提取出来的 16 类特征。

## 7. 输入通道和输出通道分别由谁决定

这一点非常重要。

输入通道数由上一层输出决定。

输出通道数由当前层卷积核个数决定。

可以这样记：

```text
输入有多少通道 -> 卷积核就要有多少层
想输出多少通道 -> 就准备多少个卷积核
```

例如：

```text
输入：3 x 32 x 32
卷积核个数：16 个
输出：16 x H_out x W_out
```

再下一层如果继续卷积，那么下一层看到的输入通道数就是 16，而不是 3。

## 8. 多通道卷积的形状关系

假设输入特征图形状是：

$$
C_{in}\times H\times W
$$

其中 $C_{in}$ 是输入通道数。

如果有 $C_{out}$ 个卷积核，那么输出通道数就是 $C_{out}$。

输出形状可以写成：

$$
C_{out}\times H_{out}\times W_{out}
$$

高和宽由卷积核大小、padding、stride 决定。

通道数由卷积核个数决定。

如果加上 batch 维度，就变成：

$$
B\times C_{out}\times H_{out}\times W_{out}
$$

## 9. 一个具体例子

假设输入是一批 RGB 图片。

单张图片形状是：

$$
3\times32\times32
$$

使用 16 个卷积核，每个卷积核空间大小是 3 x 3。

因为输入有 3 个通道，所以每个卷积核实际要覆盖 3 个输入通道。

一个卷积核的权重形状可以理解成：

$$
3\times3\times3
$$

一共有 16 个这样的卷积核。

如果输出高宽是 32 x 32，那么输出就是：

$$
16\times32\times32
$$

如果 batch size 是 64，那么一批输出就是：

$$
64\times16\times32\times32
$$

## 10. 参数量怎么计算

卷积层的参数量也和输入通道数、输出通道数有关。

一个卷积核需要覆盖所有输入通道。

如果输入通道数是 $C_{in}$，卷积核大小是 K x K，那么一个卷积核的权重数量是：

$$
C_{in}\times K\times K
$$

如果有 $C_{out}$ 个卷积核，总权重数量就是：

$$
C_{out}\times C_{in}\times K\times K
$$

如果每个卷积核还有一个偏置，总偏置数量就是：

$$
C_{out}
$$

所以卷积层总参数量通常是：

$$
C_{out}\times C_{in}\times K\times K + C_{out}
$$

## 11. 参数量例子

还是看 RGB 图片输入。

输入通道数是 3。

卷积核大小是 3 x 3。

输出通道数是 16，也就是有 16 个卷积核。

权重数量是：

$$
16\times3\times3\times3=432
$$

如果每个卷积核有一个偏置，偏置数量是：

$$
16
$$

总参数量是：

$$
432+16=448
$$

注意，这个参数量和图片高宽 32 x 32 没有直接相乘。

因为同一个卷积核会在整张图片上重复滑动使用，这就是前面说的权值共享。

## 12. 容易混淆的地方

第一个容易混淆的地方：输入通道数不等于输出通道数。

输入通道数来自输入数据或上一层。

输出通道数来自当前层卷积核个数。

第二个容易混淆的地方：一个 RGB 卷积核不是输出 3 张图。

它虽然有 3 层权重，但最后会把 3 个通道的计算结果加起来，输出 1 张特征图。

第三个容易混淆的地方：输出通道不是颜色通道。

第一层输入的 3 个通道是 R、G、B。

但卷积后的 16 个通道通常表示 16 种不同特征，而不是 16 种颜色。

## 13. 本节小结

这一节先记住这些规则：

1. 多通道卷积中，输入有几个通道，卷积核就要有几层。
2. 每个输入通道分别和对应的卷积核层做乘加。
3. 所有通道的结果加起来，得到一个输出值。
4. 一个卷积核只产生一张特征图，也就是一个输出通道。
5. 多个卷积核会产生多个输出通道。
6. 输入通道数由输入或上一层决定。
7. 输出通道数由当前层卷积核个数决定。
8. 卷积层参数量和输入通道数、输出通道数、卷积核大小有关。

## 自检问题

1. 为什么 RGB 图片输入时，一个卷积核要有 3 层？
2. 多通道卷积中，每个通道的计算结果最后怎么处理？
3. 一个卷积核会输出几张特征图？
4. 16 个卷积核会输出几个通道？
5. 输入通道数由什么决定？
6. 输出通道数由什么决定？
7. 输入是 3 x 32 x 32，使用 16 个卷积核，如果高宽不变，输出形状是多少？
8. 输入通道数是 3，卷积核大小是 3 x 3，输出通道数是 16，权重数量是多少？
9. 为什么输出的 16 个通道不能理解成 16 个颜色通道？
10. 多通道卷积和权值共享有什么关系？